In [18]:
import pandas as pd

In [19]:
dataset=pd.read_csv(r"D:\python1\Week-9.1-Machine-Learning-Data-Science-Capstone-SLA-Prediction-Project\1.Data collection\it_incident_dataset.csv")

In [20]:
dataset

,Incident_ID,Incident_Type,Priority,Assigned_Department,Location,Reported_Time,Resolved_Time,Resolution_Time_Hours,Status,Resolution_Type
0,INC100000,Network Outage,Low,Security Team,Data Center B,17-03-2024 04:00,17-03-2024 22:00,18.0,Resolved,Reboot
1,INC100001,Database Failure,Low,Database Admin,Remote Site 1,11-01-2024 20:00,12-01-2024 00:00,NaN,Resolved,Reboot
2,INC100002,Server Crash,Medium,Database Admin,Remote Site 2,10-01-2024 01:00,13-01-2024 01:00,72.0,Closed,Patch Applied
3,INC100003,Database Failure,Critical,Network Team,Remote Site 1,20-07-2024 03:00,21-07-2024 15:00,36.0,Resolved,Reboot
4,INC100004,Server Crash,Critical,Security Team,Head Office,23-02-2024 01:00,24-02-2024 05:00,28.0,Resolved,Configuration Fix
...,...,...,...,...,...,...,...,...,...,...
1195,INC101195,Application Bug,Critical,Security Team,Remote Site 2,04-10-2024 13:00,06-10-2024 12:00,47.0,Resolved,Hardware Replacement
1196,INC101196,Security Breach,Low,Network Team,Data Center B,05-01-2024 17:00,06-01-2024 18:00,25.0,Resolved,Reboot
1197,INC101197,Network Outage,Medium,IT Support,Data Center B,13-06-2024 05:00,14-06-2024 00:00,19.0,Resolved,Patch Applied
1198,INC101198,Application Bug,Medium,IT Support,Head Office,19-09-2024 17:00,22-09-2024 05:00,60.0,Closed,Patch Applied


# Clean column names

In [21]:
dataset.columns = dataset.columns.str.strip()

In [22]:
dataset.head()

,Incident_ID,Incident_Type,Priority,Assigned_Department,Location,Reported_Time,Resolved_Time,Resolution_Time_Hours,Status,Resolution_Type
0,INC100000,Network Outage,Low,Security Team,Data Center B,17-03-2024 04:00,17-03-2024 22:00,18.0,Resolved,Reboot
1,INC100001,Database Failure,Low,Database Admin,Remote Site 1,11-01-2024 20:00,12-01-2024 00:00,NaN,Resolved,Reboot
2,INC100002,Server Crash,Medium,Database Admin,Remote Site 2,10-01-2024 01:00,13-01-2024 01:00,72.0,Closed,Patch Applied
3,INC100003,Database Failure,Critical,Network Team,Remote Site 1,20-07-2024 03:00,21-07-2024 15:00,36.0,Resolved,Reboot
4,INC100004,Server Crash,Critical,Security Team,Head Office,23-02-2024 01:00,24-02-2024 05:00,28.0,Resolved,Configuration Fix


# Handling missing values

In [23]:
dataset.isnull().sum()

Incident_ID              0
Incident_Type            0
Priority                 0
Assigned_Department      0
Location                 0
Reported_Time            0
Resolved_Time            0
Resolution_Time_Hours    3
Status                   0
Resolution_Type          0
dtype: int64

# Date Handling

In [24]:
dataset["Reported_Time"] = pd.to_datetime(
    dataset["Reported_Time"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

dataset["Resolved_Time"] = pd.to_datetime(
    dataset["Resolved_Time"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

 # Feature Engineering

In [25]:
dataset["Hour"] = dataset["Reported_Time"].dt.hour
dataset["Day"] = dataset["Reported_Time"].dt.dayofweek
dataset["Month"] = dataset["Reported_Time"].dt.month

# SLA Rule ( Prediction Goal)

In [26]:
def sla_limit(priority):
    if priority == "Critical":
        return 4
    elif priority == "High":
        return 8
    elif priority == "Medium":
        return 12
    else:
        return 24

dataset["SLA_Limit"] = dataset["Priority"].apply(sla_limit)

dataset["SLA_Breached"] = (
    dataset["Resolution_Time_Hours"] > dataset["SLA_Limit"]
).astype(int)

In [27]:
dataset

,Incident_ID,Incident_Type,Priority,Assigned_Department,Location,Reported_Time,Resolved_Time,Resolution_Time_Hours,Status,Resolution_Type,Hour,Day,Month,SLA_Limit,SLA_Breached
0,INC100000,Network Outage,Low,Security Team,Data Center B,2024-03-17 04:00:00,2024-03-17 22:00:00,18.0,Resolved,Reboot,4,6,3,24,0
1,INC100001,Database Failure,Low,Database Admin,Remote Site 1,2024-01-11 20:00:00,2024-01-12 00:00:00,NaN,Resolved,Reboot,20,3,1,24,0
2,INC100002,Server Crash,Medium,Database Admin,Remote Site 2,2024-01-10 01:00:00,2024-01-13 01:00:00,72.0,Closed,Patch Applied,1,2,1,12,1
3,INC100003,Database Failure,Critical,Network Team,Remote Site 1,2024-07-20 03:00:00,2024-07-21 15:00:00,36.0,Resolved,Reboot,3,5,7,4,1
4,INC100004,Server Crash,Critical,Security Team,Head Office,2024-02-23 01:00:00,2024-02-24 05:00:00,28.0,Resolved,Configuration Fix,1,4,2,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,INC101195,Application Bug,Critical,Security Team,Remote Site 2,2024-10-04 13:00:00,2024-10-06 12:00:00,47.0,Resolved,Hardware Replacement,13,4,10,4,1
1196,INC101196,Security Breach,Low,Network Team,Data Center B,2024-01-05 17:00:00,2024-01-06 18:00:00,25.0,Resolved,Reboot,17,4,1,24,1
1197,INC101197,Network Outage,Medium,IT Support,Data Center B,2024-06-13 05:00:00,2024-06-14 00:00:00,19.0,Resolved,Patch Applied,5,3,6,12,1
1198,INC101198,Application Bug,Medium,IT Support,Head Office,2024-09-19 17:00:00,2024-09-22 05:00:00,60.0,Closed,Patch Applied,17,3,9,12,1


In [28]:
# categorical columns
cat_cols = dataset.select_dtypes(include=["object", "string"]).columns

# numerical columns
num_cols = dataset.select_dtypes(include="number").columns

# fill categorical missing values
for col in cat_cols:
    dataset[col] = dataset[col].fillna(dataset[col].mode()[0])

# fill numerical missing values
dataset[num_cols] = dataset[num_cols].apply(pd.to_numeric, errors="coerce")
dataset[num_cols] = dataset[num_cols].fillna(dataset[num_cols].median())

In [29]:
dataset[num_cols]

,Resolution_Time_Hours,Hour,Day,Month,SLA_Limit,SLA_Breached
0,18.0,4,6,3,24,0
1,35.0,20,3,1,24,0
2,72.0,1,2,1,12,1
3,36.0,3,5,7,4,1
4,28.0,1,4,2,4,1
...,...,...,...,...,...,...
1195,47.0,13,4,10,4,1
1196,25.0,17,4,1,24,1
1197,19.0,5,3,6,12,1
1198,60.0,17,3,9,12,1


In [30]:
dataset[cat_cols]

,Incident_ID,Incident_Type,Priority,Assigned_Department,Location,Status,Resolution_Type
0,INC100000,Network Outage,Low,Security Team,Data Center B,Resolved,Reboot
1,INC100001,Database Failure,Low,Database Admin,Remote Site 1,Resolved,Reboot
2,INC100002,Server Crash,Medium,Database Admin,Remote Site 2,Closed,Patch Applied
3,INC100003,Database Failure,Critical,Network Team,Remote Site 1,Resolved,Reboot
4,INC100004,Server Crash,Critical,Security Team,Head Office,Resolved,Configuration Fix
...,...,...,...,...,...,...,...
1195,INC101195,Application Bug,Critical,Security Team,Remote Site 2,Resolved,Hardware Replacement
1196,INC101196,Security Breach,Low,Network Team,Data Center B,Resolved,Reboot
1197,INC101197,Network Outage,Medium,IT Support,Data Center B,Resolved,Patch Applied
1198,INC101198,Application Bug,Medium,IT Support,Head Office,Closed,Patch Applied


# Preprocessed data

In [31]:
prep=dataset[cat_cols],dataset[num_cols]
preprocessed=pd.concat(prep,axis=1)

In [32]:
preprocessed

,Incident_ID,Incident_Type,Priority,Assigned_Department,Location,Status,Resolution_Type,Resolution_Time_Hours,Hour,Day,Month,SLA_Limit,SLA_Breached
0,INC100000,Network Outage,Low,Security Team,Data Center B,Resolved,Reboot,18.0,4,6,3,24,0
1,INC100001,Database Failure,Low,Database Admin,Remote Site 1,Resolved,Reboot,35.0,20,3,1,24,0
2,INC100002,Server Crash,Medium,Database Admin,Remote Site 2,Closed,Patch Applied,72.0,1,2,1,12,1
3,INC100003,Database Failure,Critical,Network Team,Remote Site 1,Resolved,Reboot,36.0,3,5,7,4,1
4,INC100004,Server Crash,Critical,Security Team,Head Office,Resolved,Configuration Fix,28.0,1,4,2,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,INC101195,Application Bug,Critical,Security Team,Remote Site 2,Resolved,Hardware Replacement,47.0,13,4,10,4,1
1196,INC101196,Security Breach,Low,Network Team,Data Center B,Resolved,Reboot,25.0,17,4,1,24,1
1197,INC101197,Network Outage,Medium,IT Support,Data Center B,Resolved,Patch Applied,19.0,5,3,6,12,1
1198,INC101198,Application Bug,Medium,IT Support,Head Office,Closed,Patch Applied,60.0,17,3,9,12,1


In [33]:
preprocessed.isnull().sum()

Incident_ID              0
Incident_Type            0
Priority                 0
Assigned_Department      0
Location                 0
Status                   0
Resolution_Type          0
Resolution_Time_Hours    0
Hour                     0
Day                      0
Month                    0
SLA_Limit                0
SLA_Breached             0
dtype: int64

# Save preprocessed data

In [34]:
preprocessed.to_csv("preprocessed_data.csv", index=False)